# Chapter 18: conditional variance and forecast clocks

Check hand formulas, then freshly call analyze_constructed() and save constructed_simulation.csv under RUN_DIR. The default does not run the French crisis window or generate its historical figures. Returns are decimal monthly observations; variances use squared return units. Fixed-mean innovations make these recursions self-contained with Chapter 16's time-series background; Chapter 17's unit-root and cointegration results are not inputs.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch18_volatility_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(ROOT / "code"))

import ch18_volatility as workflow
from quant_course.volatility import (
    GARCHFit, GARCHParameters, ewma_variance_path, garch_variance_forecast,
    garch_variance_path, realized_variance,
)
print("Project root:", ROOT)

## 1. Realized variation versus squared net movement

For subinterval log returns (0.01, −0.02, 0.03), the sum of squares is 0.0014, while the square of their sum is 0.0004. The EWMA/ARCH/GARCH exercise reinterprets these values as consecutive monthly innovations, a different target. Compare their paths and multistep conditional forecasts.

In [ ]:
parts = [0.01, -0.02, 0.03]
rv = realized_variance(parts)
net_squared = sum(parts) ** 2
assert abs(rv - 0.0014) < 1e-12
assert abs(net_squared - 0.0004) < 1e-12
idx = pd.period_range("2000-01", periods=3, freq="M")
shocks = pd.Series(parts, index=idx)
params = GARCHParameters(omega=0.00001, alpha=0.1, beta=0.8)
ewma = ewma_variance_path(shocks, decay=0.9, initial_variance=0.0004)
garch = garch_variance_path(shocks, params, initial_variance=0.0004)
arch = garch_variance_path(shocks, GARCHParameters(0.00001, 0.1, 0.0), initial_variance=0.0004)
toy_fit = GARCHFit(params, 0.0, pd.Period("2000-03", freq="M"), 60, 0.0004, 0.0, 1, garch)
multi_step = garch_variance_forecast(toy_fit, steps=3)
assert abs(ewma.iloc[0] - 0.0004) < 1e-12
assert abs(ewma.iloc[1] - 0.00037) < 1e-12
assert abs(ewma.iloc[2] - 0.000373) < 1e-12
assert all(abs(actual - expected) < 1e-12 for actual, expected in zip(
    arch.tolist(), [0.0004, 0.00002, 0.00005]))
assert all(abs(actual - expected) < 1e-12 for actual, expected in zip(
    garch.tolist(), [0.0004, 0.00034, 0.000322]))
print({"RV": rv, "net_return_squared": net_squared,
       "EWMA h1:h3": ewma.tolist(), "ARCH h1:h3": arch.tolist(),
       "GARCH h1:h3": garch.tolist(),
       "GARCH long-run h": params.unconditional_variance,
       "expected one-month h for horizons 1:3": multi_step.tolist()})

Opposite directions cancel in net movement, but not in the sum of squares. EWMA and GARCH return variance paths; their square roots are return standard deviations. A period-end innovation updates the following period rather than its own pre-observation forecast.

In [ ]:
summary, simulation = workflow.analyze_constructed()
simulation.to_csv(RUN_DIR / "constructed_simulation.csv")
print(summary)
assert summary["simulation"]["observations"] == 360


## 2. Constructed path and skipped historical window

The executable cell generates 360 constructed observations and writes their CSV. Since the French branch is disabled, no market ZIP/CSV audit or 204-month historical fit occurs here. The archived study trained on 1990–2006 and evaluated 36 months in 2007–2009. A target month's squared innovation becomes available only after that month ends.

In [ ]:
if RUN_FRENCH:
    actual = pd.read_csv(ROOT / "outputs/ch18/ch18_actual_forecasts.csv", parse_dates=False)
    actual.head(), actual.tail(), len(actual)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 3. Losses and evidence

Minimum QLIKE against a noisy market proxy does not identify a tradable champion. Mean misspecification can bias that proxy, and a 36-month crisis slice does not establish a significant winner. Causal recursion within the algorithm does not restore missing source vintages. Without positions and fills, net strategy return is inapplicable rather than costless. The constructed simulation has known true_variance; finite burn-in reduces initialization effects without ensuring exact stationary sampling. The historical gallery stays skipped.

In [ ]:
simulation_summary = summary["simulation"]
simulation_summary